In [4]:
import pandas as pd
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

data = pd.read_csv('/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv',encoding='latin-1')
data = data[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
data['label'] = data['label'].map({'ham': 0, 'spam': 1})

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

data['clean_text'] = data['text'].apply(preprocess)

vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(data['clean_text'])
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

feature_names = vectorizer.get_feature_names_out()
coefs = pd.Series(model.coef_[0], index=feature_names)
print("Top Spam Words:\n", coefs.sort_values(ascending=False).head(10))

test_msg = "Congrats! You won $1000."
test_msg_clean = preprocess(test_msg)
test_vec = vectorizer.transform([test_msg_clean])
pred = model.predict(test_vec)

print(f"\nPrediction for '{test_msg}': {'Spam' if pred[0] == 1 else 'Not Spam'}")

Top Spam Words:
 txt       4.602971
call      4.212298
stop      3.475956
claim     3.353359
text      3.300068
free      3.283940
reply     3.122549
mobile    3.013742
from      2.831425
now       2.724235
dtype: float64

Prediction for 'Congrats! You won $1000.': Not Spam
